# Types of Crossmatches

In this tutorial, we will:

- explain what `how='inner'` (the default) and `how='left'` mean when crossmatching catalogs
- demonstrate how each join type affects the row count and column values in the result
- give guidance on when to choose each option

In [1]:
import warnings

import pandas as pd

import lsdb

warnings.filterwarnings("ignore")

## 1. Set up two synthetic catalogs

We create two small catalogs from plain pandas DataFrames to keep this example self-contained.

`catalog_a` has **6 sources** spread across a small strip of sky.
The first three sources sit very close to sources in `catalog_b`;
the last three have no nearby counterpart.

In [2]:
catalog_a_data = pd.DataFrame({
    "ra":        [10.0, 10.5, 11.0, 11.5, 12.0, 12.5],
    "dec":       [20.0, 20.0, 20.0, 20.0, 20.0, 20.0],
    "source_id": [1,    2,    3,    4,    5,    6   ],
    "mag_a":     [15.0, 16.0, 17.0, 18.0, 19.0, 20.0],
})
catalog_a_data

,ra,dec,source_id,mag_a
0,10.0,20.0,1,15.0
1,10.5,20.0,2,16.0
2,11.0,20.0,3,17.0
3,11.5,20.0,4,18.0
4,12.0,20.0,5,19.0
5,12.5,20.0,6,20.0


`catalog_b` has **3 sources**, each located about 0.5 arcsec from one of the first three sources in `catalog_a`.

In [3]:
catalog_b_data = pd.DataFrame({
    "ra":        [10.0001, 10.5001, 11.0001],
    "dec":       [20.0001, 20.0001, 20.0001],
    "object_id": [101,     102,     103    ],
    "mag_b":     [15.1,    16.1,    17.1   ],
})
catalog_b_data

,ra,dec,object_id,mag_b
0,10.0001,20.0001,101,15.1
1,10.5001,20.0001,102,16.1
2,11.0001,20.0001,103,17.1


Convert both DataFrames to LSDB `Catalog` objects.

In [4]:
catalog_a = lsdb.from_dataframe(catalog_a_data, catalog_name="catalog_a", ra_column="ra", dec_column="dec")
catalog_b = lsdb.from_dataframe(catalog_b_data, catalog_name="catalog_b", ra_column="ra", dec_column="dec")

## 2. What's the difference between left and right catalogs?

When you write

```python
catalog_a.crossmatch(catalog_b, ...)
```

- **`catalog_a` is always the left catalog** — the catalog you call `.crossmatch()` on.
- **`catalog_b` is always the right catalog** — the catalog you pass as the first argument.

The distinction matters because the crossmatch algorithm iterates over every row in the *left* catalog and searches for its nearest neighbor in the *right* catalog. This means that from the left catalog there will never be duplicates, each row in the left catalog will appear at most once in the output. But from the right catalog, there can be duplicates — if multiple left-catalog rows are close to the same right-catalog row, they will all match to it and appear in the result.

In addition, rows in the left catalog that have no counterpart in the right catalog within the search radius are handled differently depending on `how`.

If you're using the lsdb.crossmatch function, the left and right catalogs are determined by the order of the arguments:

```python
result = lsdb.crossmatch(catalog_a, catalog_b, ...)
```

## 3. Inner crossmatch (`how='inner'`, the default)

`how='inner'` keeps **only the rows where a match was found** in both catalogs.
Any left-catalog source that falls outside the search radius of every right-catalog source
is silently dropped from the result.

This is the default behavior, so `catalog_a.crossmatch(catalog_b, radius_arcsec=1.0)` and
`catalog_a.crossmatch(catalog_b, radius_arcsec=1.0, how='inner')` are equivalent.

In [5]:
inner_result = catalog_a.crossmatch(
    catalog_b,
    radius_arcsec=1.0,
    how="inner",
    suffix_method="overlapping_columns",
    log_changes=False,
).compute().sort_values("source_id")

inner_result

,ra_catalog_a,dec_catalog_a,source_id,mag_a,ra_catalog_b,dec_catalog_b,object_id,mag_b,_dist_arcsec
_healpix_29,,,,,,,,,
1397760956975030485,10.0,20.0,1,15.0,10.0001,20.0001,101,15.1,0.494004
1397663462289472703,10.5,20.0,2,16.0,10.5001,20.0001,102,16.1,0.494004
1400656817198057343,11.0,20.0,3,17.0,11.0001,20.0001,103,17.1,0.494004


In [6]:
print(f"catalog_a rows : {len(catalog_a_data)}")
print(f"catalog_b rows : {len(catalog_b_data)}")
print(f"inner result   : {len(inner_result)}  (sources 4, 5, 6 were dropped \u2014 no match found)")

catalog_a rows : 6
catalog_b rows : 3
inner result   : 3  (sources 4, 5, 6 were dropped — no match found)


## 4. Left crossmatch (`how='left'`)

`how='left'` keeps **every row from the left catalog**, regardless of whether a match was found.
For left-catalog sources that have no counterpart in the right catalog, all right-catalog columns
in that row are filled with `<NA>`.

The row count of the result equals the row count of the left catalog (assuming `n_neighbors=1`).

In [7]:
left_result = catalog_a.crossmatch(
    catalog_b,
    radius_arcsec=1.0,
    how="left",
    suffix_method="overlapping_columns",
    log_changes=False,
).compute().sort_values("source_id")

left_result

,ra_catalog_a,dec_catalog_a,source_id,mag_a,ra_catalog_b,dec_catalog_b,object_id,mag_b,_dist_arcsec
_healpix_29,,,,,,,,,
1397760956975030485,10.0,20.0,1,15.0,10.0001,20.0001,101,15.1,0.494004
1397663462289472703,10.5,20.0,2,16.0,10.5001,20.0001,102,16.1,0.494004
1400656817198057343,11.0,20.0,3,17.0,11.0001,20.0001,103,17.1,0.494004
1400640342732229567,11.5,20.0,4,18.0,<NA>,<NA>,<NA>,<NA>,<NA>
1394637330291997823,12.0,20.0,5,19.0,<NA>,<NA>,<NA>,<NA>,<NA>
1394677858677559530,12.5,20.0,6,20.0,<NA>,<NA>,<NA>,<NA>,<NA>


In [8]:
print(f"catalog_a rows : {len(catalog_a_data)}")
print(f"catalog_b rows : {len(catalog_b_data)}")
print(f"left result    : {len(left_result)}  (all catalog_a sources preserved)")

catalog_a rows : 6
catalog_b rows : 3
left result    : 6  (all catalog_a sources preserved)


### 4.1 Identifying unmatched sources

Because unmatched rows have `<NA>` in all right-catalog columns, you can filter
them with a simple null check on any right-catalog column — for example `object_id`:

In [9]:
unmatched = left_result[left_result["object_id"].isna()]
print("Unmatched source_ids:", unmatched["source_id"].to_list())
print(f"Match fraction: {len(left_result) - len(unmatched)}/{len(left_result)} = {(1 - len(unmatched)/len(left_result)):.1%}")

Unmatched source_ids: [4, 5, 6]
Match fraction: 50.0%


## 5. When to choose each

| | `how='inner'` (default) | `how='left'` |
|---|---|---|
| **Result rows** | One per matched pair | One per left-catalog source |
| **Unmatched left sources** | Dropped silently | Kept, right columns are `<NA>` |
| **Typical use** | You only care about confirmed matches | You need to account for every left-catalog source |

**Use `how='inner'`** when you only need sources that appear in both catalogs — for example,
comparing photometry for objects observed by two telescopes, where sources not seen by both are irrelevant.

**Use `how='left'`** when you need to preserve every source from your primary (left) catalog.
Common scenarios include:

- **Completeness analysis**: computing what fraction of your primary sample has a counterpart at another wavelength.
- **Upper-limit studies**: you need a row per source even when no match exists, so you can fill in flux upper limits.
- **Flagging**: you want to add a "matched" boolean column and keep the full primary catalog.

```python
# Use left join to flag matched sources and keep everything
result = catalog_a.crossmatch(catalog_b, radius_arcsec=1.0, how="left")
result_df = result.compute()
result_df["has_match"] = result_df["object_id"].notna()
```

> **Default is `'inner'`**: If the result of `.crossmatch()` has fewer rows than your left catalog,
> the missing rows are unmatched sources that were silently dropped.
> Switch to `how='left'` to see them.

## About

**Author(s):** Sean McGuire

**Last updated on**: 18 May 2026

If you use `lsdb` for published research, please cite following [instructions](https://docs.lsdb.io/en/stable/citation.html).